# 02 — CLIP Embedding Space Analysis

**Owner:** Model Engineer (ME-4, Sprint 2)

**Goal:** Visualize the CLIP embedding space using UMAP/t-SNE to verify that:
1. Images of the same class cluster together
2. Text queries land near their matching image clusters
3. The encoder produces meaningful, separable representations

---

## 1. Setup & Imports

In [ ]:
# Install UMAP if not available
# !pip install umap-learn

import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

# Add project root to path so we can import our modules
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.model.clip_encoder import CLIPEncoder
from src.model.embeddings_cache import EmbeddingsCache

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 100

print("Imports loaded successfully ✅")

## 2. Load Encoder & Data

We load the CLIP encoder and either:
- **Option A:** Load cached embeddings from `data/index/` (if indexing pipeline has run)
- **Option B:** Encode images on-the-fly from `data/images/`

In [ ]:
# Initialize encoder
encoder = CLIPEncoder()
print(f"Model: {encoder.model_name}")
print(f"Device: {encoder.device}")

In [ ]:
# --- Option A: Load from cache (fast) ---
CACHE_PATH = PROJECT_ROOT / "data" / "index" / "embeddings"
METADATA_PATH = PROJECT_ROOT / "data" / "metadata.json"

USE_CACHE = Path(str(CACHE_PATH) + ".npy").exists()

if USE_CACHE:
    print("Loading embeddings from cache...")
    embeddings, ids = EmbeddingsCache.load(CACHE_PATH)
    
    # Load metadata to get labels
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)
    
    # Build id -> label mapping
    id_to_label = {item["path"]: item.get("label", "unknown") for item in metadata}
    labels = [id_to_label.get(img_id, "unknown") for img_id in ids]
    
    print(f"Loaded {len(ids)} embeddings from cache ✅")

else:
    print("No cache found. Will encode images on-the-fly in the next cell.")
    print(f"(Expected cache at: {CACHE_PATH}.npy)")

In [ ]:
# --- Option B: Encode from image directory (slower, used when no cache) ---
if not USE_CACHE:
    IMAGE_DIR = PROJECT_ROOT / "data" / "images"
    
    if not IMAGE_DIR.exists():
        raise FileNotFoundError(
            f"No images found at {IMAGE_DIR}. "
            f"Ask the Data Engineer to prepare the dataset first."
        )
    
    # Collect image paths and extract labels from folder names
    # Expected structure: data/images/cat/001.jpg, data/images/dog/002.jpg
    image_paths = []
    labels = []
    
    for label_dir in sorted(IMAGE_DIR.iterdir()):
        if not label_dir.is_dir():
            continue
        for img_path in sorted(label_dir.glob("*")):
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
                image_paths.append(str(img_path))
                labels.append(label_dir.name)
    
    print(f"Found {len(image_paths)} images across {len(set(labels))} classes")
    
    # Encode all images
    print("Encoding images (this may take a few minutes)...")
    pil_images = [Image.open(p).convert("RGB") for p in image_paths]
    embeddings = encoder.encode_images_batch(pil_images, batch_size=32)
    ids = image_paths
    
    print(f"Encoded {embeddings.shape[0]} images → shape {embeddings.shape} ✅")

In [ ]:
# Summary statistics
unique_labels = sorted(set(labels))
print(f"Total images: {len(labels)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Number of classes: {len(unique_labels)}")
print(f"\nClass distribution:")
for label in unique_labels:
    count = labels.count(label)
    print(f"  {label}: {count} images")

## 3. Dimensionality Reduction: UMAP

UMAP (Uniform Manifold Approximation and Projection) reduces our 512-d vectors to 2D for visualization.

**Why UMAP over t-SNE?**
- Faster (minutes vs hours on large datasets)
- Preserves both local AND global structure
- More stable across runs with `random_state`

In [ ]:
try:
    import umap
    
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,       # balance local vs global structure
        min_dist=0.1,         # how tightly points cluster
        metric="cosine",      # match how FAISS compares vectors
        random_state=42       # reproducibility
    )
    
    print("Running UMAP dimensionality reduction...")
    embeddings_2d = reducer.fit_transform(embeddings)
    print(f"Reduced: {embeddings.shape} → {embeddings_2d.shape} ✅")
    REDUCTION_METHOD = "UMAP"

except ImportError:
    print("UMAP not installed. Falling back to t-SNE...")
    print("(Install UMAP for better results: pip install umap-learn)")
    
    from sklearn.manifold import TSNE
    
    reducer = TSNE(
        n_components=2,
        perplexity=30,
        random_state=42,
        n_iter=1000
    )
    
    print("Running t-SNE dimensionality reduction...")
    embeddings_2d = reducer.fit_transform(embeddings)
    print(f"Reduced: {embeddings.shape} → {embeddings_2d.shape} ✅")
    REDUCTION_METHOD = "t-SNE"

## 4. Visualization: Cluster by Class

If CLIP is working well, we should see:
- Distinct, separated clusters for each class
- Related classes (e.g., "cat" and "dog") closer together than unrelated ones (e.g., "cat" and "car")

In [ ]:
# Create color palette
palette = sns.color_palette("husl", n_colors=len(unique_labels))
label_to_color = {label: palette[i] for i, label in enumerate(unique_labels)}
colors = [label_to_color[label] for label in labels]

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

for i, label in enumerate(unique_labels):
    mask = [l == label for l in labels]
    indices = [j for j, m in enumerate(mask) if m]
    ax.scatter(
        embeddings_2d[indices, 0],
        embeddings_2d[indices, 1],
        c=[palette[i]],
        label=label,
        alpha=0.6,
        s=40,
        edgecolors="white",
        linewidth=0.5
    )

ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
ax.set_title(f"CLIP Embedding Space ({REDUCTION_METHOD} projection)", fontsize=16, fontweight="bold")
ax.set_xlabel(f"{REDUCTION_METHOD} Dimension 1")
ax.set_ylabel(f"{REDUCTION_METHOD} Dimension 2")
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "notebooks" / "embedding_space_clusters.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Plot saved to notebooks/embedding_space_clusters.png ✅")

## 5. Text Query Overlay

Project text queries into the same 2D space to see if they land near the correct image clusters.

In [ ]:
# Define text queries — one per class
text_queries = [f"a photo of a {label}" for label in unique_labels]

# Encode text queries
text_embeddings = np.array([encoder.encode_text(q) for q in text_queries])
print(f"Encoded {len(text_queries)} text queries → shape {text_embeddings.shape}")

# Project text embeddings into the same 2D space
# Combine with image embeddings and re-run UMAP to get a consistent projection
all_embeddings = np.vstack([embeddings, text_embeddings])

try:
    import umap
    reducer_combined = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
except ImportError:
    from sklearn.manifold import TSNE
    reducer_combined = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)

print("Running combined projection...")
all_2d = reducer_combined.fit_transform(all_embeddings)

image_2d = all_2d[:len(embeddings)]
text_2d = all_2d[len(embeddings):]

print("Done ✅")

In [ ]:
# Plot images + text queries together
fig, ax = plt.subplots(figsize=(14, 10))

# Plot image points (faded)
for i, label in enumerate(unique_labels):
    mask = [l == label for l in labels]
    indices = [j for j, m in enumerate(mask) if m]
    ax.scatter(
        image_2d[indices, 0],
        image_2d[indices, 1],
        c=[palette[i]],
        label=f"{label} (images)",
        alpha=0.3,
        s=30
    )

# Plot text queries (bold stars)
for i, (label, query) in enumerate(zip(unique_labels, text_queries)):
    ax.scatter(
        text_2d[i, 0],
        text_2d[i, 1],
        c=[palette[i]],
        marker="*",
        s=400,
        edgecolors="black",
        linewidth=1.5,
        zorder=5
    )
    ax.annotate(
        query,
        (text_2d[i, 0], text_2d[i, 1]),
        textcoords="offset points",
        xytext=(10, 10),
        fontsize=9,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8)
    )

ax.set_title(f"CLIP Embedding Space: Images + Text Queries ({REDUCTION_METHOD})", fontsize=16, fontweight="bold")
ax.set_xlabel(f"{REDUCTION_METHOD} Dimension 1")
ax.set_ylabel(f"{REDUCTION_METHOD} Dimension 2")
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "notebooks" / "embedding_text_overlay.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Plot saved to notebooks/embedding_text_overlay.png ✅")

## 6. Inter-class Similarity Heatmap

Compute average cosine similarity between class centroids to see which classes CLIP considers most similar.

In [ ]:
# Compute class centroids (mean embedding per class)
centroids = {}
for label in unique_labels:
    indices = [i for i, l in enumerate(labels) if l == label]
    centroid = embeddings[indices].mean(axis=0)
    # Re-normalize the centroid to unit length
    centroid = centroid / np.linalg.norm(centroid)
    centroids[label] = centroid

# Build similarity matrix
n_classes = len(unique_labels)
sim_matrix = np.zeros((n_classes, n_classes))

for i, label_i in enumerate(unique_labels):
    for j, label_j in enumerate(unique_labels):
        # Cosine similarity = dot product of unit vectors
        sim_matrix[i, j] = np.dot(centroids[label_i], centroids[label_j])

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    sim_matrix,
    xticklabels=unique_labels,
    yticklabels=unique_labels,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    vmin=0.5,
    vmax=1.0,
    ax=ax
)
ax.set_title("Inter-class Cosine Similarity (CLIP Centroids)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "notebooks" / "class_similarity_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Heatmap saved to notebooks/class_similarity_heatmap.png ✅")

## 7. Embedding Quality Metrics

In [ ]:
# Intra-class vs Inter-class similarity
intra_sims = []  # similarity between images of the SAME class
inter_sims = []  # similarity between images of DIFFERENT classes

# Sample pairs to avoid O(N^2) computation on large datasets
rng = np.random.RandomState(42)
n_samples = min(5000, len(labels) * (len(labels) - 1) // 2)

for _ in range(n_samples):
    i, j = rng.choice(len(labels), size=2, replace=False)
    sim = np.dot(embeddings[i], embeddings[j])  # cosine sim (vectors are normalized)
    
    if labels[i] == labels[j]:
        intra_sims.append(sim)
    else:
        inter_sims.append(sim)

print(f"Intra-class similarity (same class):     mean={np.mean(intra_sims):.4f}, std={np.std(intra_sims):.4f}")
print(f"Inter-class similarity (different class): mean={np.mean(inter_sims):.4f}, std={np.std(inter_sims):.4f}")
print(f"Gap (higher is better):                   {np.mean(intra_sims) - np.mean(inter_sims):.4f}")

# Plot distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(intra_sims, bins=50, alpha=0.7, label=f"Intra-class (μ={np.mean(intra_sims):.3f})", color="#2ecc71")
ax.hist(inter_sims, bins=50, alpha=0.7, label=f"Inter-class (μ={np.mean(inter_sims):.3f})", color="#e74c3c")
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Count")
ax.set_title("Embedding Quality: Intra-class vs Inter-class Similarity", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "notebooks" / "similarity_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Analysis complete!")
print("If the green (intra) distribution is clearly to the RIGHT of the red (inter),")
print("your embeddings are good — CLIP can distinguish between classes.")